# Build and Inspect SplitDatasetContainer

This tutorial shows how to construct a `SplitDatasetContainer` from NumPy arrays and inspect it using the three standard views: **Container View**, **Machine View**, and **Analyst View**.

## Three Views of the Data

The `SplitDatasetContainer` stores data in a column-oriented format but allows you to pivot it depending on your needs.

### A. Container View (Standard Access)
*How the data looks when you access it directly via dot-notation.*

```python
container.features.train
# └── List[ndarray] (one per unit)
#     ├── Unit 0: Array(shape=(100, 3))
#     ├── Unit 1: Array(shape=(80, 3))
#     └── ...

container.target.train
# └── List[ndarray] (one per unit)
#     ├── Unit 0: Array(shape=(100, 1))
#     ├── Unit 1: Array(shape=(80, 1))
#     └── ...
```

### B. Machine View (to_split_dict)
*Used by PyTorch/Hydra. Organized by Split → Field → List of Units*

```python
# Output of container.to_split_dict()
{
    "train": {
        "features": [Array_Unit0, Array_Unit1, ...],
        "target":   [Array_Unit0, Array_Unit1, ...]
    },
    "test": {
        "features": [Array_Unit0, ...],
        "target":   [Array_Unit0, ...]
    }
}
```

### C. Analyst View (group_by_split)
*Used for plotting/debugging. Organized by Split → Unit Object*

```python
# Output of container.group_by_split()
{
    "train": [
        BaseDataObjectWithMetadata(features=Array_Unit0, target=Array_Unit0, metadata={...}),
        BaseDataObjectWithMetadata(features=Array_Unit1, target=Array_Unit1, metadata={...})
    ],
    "test": [...]
}
```

## Build the Container

Construct a container from NumPy arrays. The constructor accepts `features` and `target` as dicts mapping split names to lists of arrays (one array per unit).

In [ ]:
import numpy as np
from picid.data.data_objects import SplitDatasetContainer

arr1 = np.random.randn(100, 3).astype(np.float32)
arr2 = np.random.randn(80, 3).astype(np.float32)
t1 = np.zeros((100, 1), dtype=np.float32)
t2 = np.ones((80, 1), dtype=np.float32)

container = SplitDatasetContainer(
    features={"train": [arr1, arr2], "test": [arr1[:50]]},
    target={"train": [t1, t2], "test": [t1[:50]]},
)

In [ ]:
# Access via dot-notation (Container View)
print("Train units:", len(container.features.train))
print("First unit features shape:", container.features.train[0].shape)
print("First unit target shape:", container.target.train[0].shape)

In [ ]:
# Machine View: to_split_dict() — Split → Field → List of Units
d = container.to_split_dict()
print("Splits:", list(d.keys()))
print("Train keys:", list(d["train"].keys()))
print("Train features (list of units):", len(d["train"]["features"]), "units")

In [ ]:
# Analyst View: group_by_split() — Split → List of Unit Objects
g = container.group_by_split()
print("Train units (as objects):", len(g["train"]))
unit0 = g["train"][0]
print("Unit 0 features shape:", unit0.features.shape)
print("Unit 0 target shape:", unit0.target.shape)

## Type Safety and Structure Validation

The container enforces:

1. **Strict Typing:** Fields hold only `pandas.DataFrame`, `pandas.Series`, `numpy.ndarray`, or `awkward.Array` (or lists of these). Other types raise `TypeError`.
2. **Structure Validation:** For each split, all fields (e.g. `features`, `target`) must have the same number of units. Call `container.validate()` to check.
3. **Metadata Persistence:** Column names and unit identifiers travel with the data.

In [ ]:
container.validate()  # Raises ValueError if inconsistent
print("Validation passed.")